In [ ]:
!pip install fastapi uvicorn sentence-transformers faiss-cpu \
             beautifulsoup4 requests rank_bm25 numpy aiohttp nest_asyncio pyngrok nltk

In [ ]:
from google.colab import files

# This will open a file picker — upload your backend.py and frontend.html
uploaded = files.upload()

In [ ]:
import nest_asyncio, uvicorn, threading, time, requests
nest_asyncio.apply()

def run():
    uvicorn.run("backend:app", host="0.0.0.0", port=8000)

t = threading.Thread(target=run, daemon=True)
t.start()

# Wait until server actually responds
print("Waiting for server to start...")
for i in range(60):
    try:
        r = requests.get("http://localhost:8000/health")
        if r.status_code == 200:
            print("Server ready!", r.json())
            break
    except:
        time.sleep(3)
        print(f"  still loading... ({i*3}s)")

from pyngrok import ngrok
ngrok.set_auth_token("3Co2UOUawa6m3LCwwvNLrWtXRjj_h78GiVaWgfS96a1ibUTb")
public_url = ngrok.connect(8000)
print("ngrok URL:", public_url)

In [ ]:
# Copy the URL printed above (without quotes), paste it below
NGROK_URL =  "https://uncut-platypus-carrousel.ngrok-free.dev"

with open("frontend.html", "r") as f:
    html = f.read()

# Replace whatever URL is currently in the file
import re
html = re.sub(r'const API = ".*?"', f'const API = "{NGROK_URL}"', html)

with open("frontend_ready.html", "w") as f:
    f.write(html)

from google.colab import files
files.download("frontend_ready.html")
print("Done — open the downloaded frontend_ready.html in your browser")

In [ ]:
import requests, time

BACKEND = NGROK_URL  # uses the variable from Cell 4

def get_category_members(category, limit=300):
    r = requests.get("https://en.wikipedia.org/w/api.php", params={
        "action": "query", "list": "categorymembers",
        "cmtitle": f"Category:{category}",
        "cmlimit": limit, "cmtype": "page", "format": "json"
    }, headers={"User-Agent": "TheExplorer/1.0"})
    return [m["title"] for m in r.json()["query"]["categorymembers"]]

topics = (
    get_category_members("Animals", 300) +
    get_category_members("Countries", 200) +
    get_category_members("Cities", 200) +
    get_category_members("Science", 200) +
    get_category_members("Sports", 200) +
    get_category_members("History", 200) +
    get_category_members("Technology", 200) +
    get_category_members("Geography", 200) +
    get_category_members("Food and drink", 200) +
    get_category_members("Music", 200)
)

topics = list(dict.fromkeys(topics))  # deduplicate
print(f"Total topics: {len(topics)}")

r = requests.post(f"{BACKEND}/index/wikipedia", json={
    "topics": topics,
    "max_chars": 5000
})
print(r.json())

In [ ]:
while True:
    s = requests.get(f"{BACKEND}/stats").json()
    print(f"Docs indexed: {s['docs_indexed']}", end="\r")
    time.sleep(5)